# Improving Performance of Neural Networks

> Neural networks rarely perform well out of the box. This guide covers the most important techniques to diagnose and fix common problems — from slow training to overfitting to unstable gradients.

---

## Overview

Improving a neural network's performance generally falls into two categories:

1. **Fine-tuning hyperparameters** — learning rate, batch size, number of layers, neurons, etc.
2. **Solving structural training problems** — vanishing/exploding gradients, insufficient data, slow training, and overfitting

---

## 1. Fine-Tuning Neural Network Parameters

### 1.1 Learning Rate

The learning rate $\alpha$ is the most critical hyperparameter. Too high → diverges. Too low → trains forever.

| Strategy | Description |
|---|---|
| **Learning Rate Scheduling** | Reduce $\alpha$ over time (step decay, cosine annealing) |
| **Warm-up** | Start with a small $\alpha$, ramp up, then decay |
| **Adaptive Optimizers** | Use Adam, RMSProp — they adjust $\alpha$ per parameter automatically |

```python
# PyTorch: cosine annealing schedule
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
```

---

### 1.2 Number of Layers and Neurons

- **Too few layers/neurons** → underfitting (model is too simple)
- **Too many layers/neurons** → overfitting and slower training

**Rule of thumb:** Start small, increase capacity until validation loss stops improving.

---

### 1.3 Batch Size

- Smaller batches → noisier updates, can generalize better
- Larger batches → faster training, but may converge to sharp minima

Always prefer powers of 2 (32, 64, 128) for hardware efficiency.

---

### 1.4 Number of Epochs

Train long enough to converge, but use **early stopping** to avoid overfitting:

```python
# Stop training if validation loss doesn't improve for 10 epochs
EarlyStopping(monitor='val_loss', patience=10)
```

---

### 1.5 Weight Initialization

Poor initialization can cause gradients to vanish or explode from the very first pass.

| Activation | Recommended Initialization |
|---|---|
| ReLU | He (Kaiming) Initialization |
| Sigmoid / Tanh | Xavier (Glorot) Initialization |

```python
# PyTorch: He initialization
nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
```

---

## 2. Solving Common Training Problems

---

### 2.1 Vanishing & Exploding Gradients

During backpropagation, gradients are multiplied through each layer. In deep networks, they can shrink to near-zero (**vanishing**) or grow uncontrollably (**exploding**), making training unstable or impossible.

**Symptoms:**
- Loss barely changes (vanishing)
- Loss becomes `NaN` (exploding)
- Early layers learn much slower than later layers

**Solutions:**

| Problem | Fix |
|---|---|
| Vanishing gradients | Use **ReLU** or **LeakyReLU** instead of Sigmoid/Tanh |
| Vanishing gradients | **Batch Normalization** between layers |
| Vanishing gradients | **Residual connections** (skip connections) as in ResNet |
| Exploding gradients | **Gradient clipping** — cap gradients at a max norm |
| Both | Better **weight initialization** (He / Xavier) |

```python
# Gradient clipping in PyTorch
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

---

### 2.2 Not Enough Data

Deep networks are data-hungry. Insufficient data leads to poor generalization.

**Solutions:**

#### Data Augmentation
Artificially expand your dataset by applying transformations:

```python
# torchvision augmentation pipeline
transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])
```

#### Transfer Learning
Use a model pre-trained on a large dataset (e.g., ImageNet) and fine-tune it on your smaller dataset. You leverage millions of examples without collecting them yourself.

```python
# Load pretrained ResNet, replace final layer
model = torchvision.models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
```

#### Synthetic Data Generation
Use GANs or simulation to generate realistic training samples when real data is scarce.

---

### 2.3 Slow Training

**Causes and fixes:**

| Cause | Solution |
|---|---|
| Low learning rate | Increase $\alpha$ or use a scheduler |
| Suboptimal optimizer | Switch from SGD to **Adam** or **AdamW** |
| Small batch size | Increase batch size (within memory limits) |
| No GPU utilization | Verify model and data are on GPU (`model.cuda()`) |
| Data loading bottleneck | Use multiple DataLoader workers (`num_workers=4`) |
| Unnecessary computation | Use **mixed precision training** (FP16) |

```python
# Mixed precision training (PyTorch)
scaler = torch.cuda.amp.GradScaler()
with torch.cuda.amp.autocast():
    output = model(input)
    loss = criterion(output, target)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```

---

### 2.4 Overfitting

Overfitting occurs when the model memorizes the training data and fails to generalize to unseen data. Training accuracy is high, but validation accuracy is low.

**Solutions:**

#### Dropout
Randomly deactivates neurons during training, forcing the network to learn redundant representations.

```python
nn.Dropout(p=0.5)  # 50% of neurons dropped per forward pass
```

#### L1 / L2 Regularization
Penalizes large weights, discouraging the model from over-relying on any single feature.

- **L2 (Weight Decay):** $L_{total} = L + \lambda \sum \theta^2$ — most common
- **L1:** Encourages sparsity (some weights go to exactly 0)

```python
# L2 regularization via optimizer weight_decay
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
```

#### Batch Normalization
Normalizes activations within each mini-batch, acting as a mild regularizer while also speeding up training.

```python
nn.BatchNorm1d(num_features=128)
```

#### Early Stopping
Monitor validation loss and stop training when it starts to increase — before the model overfits.

#### Increase Data
More data is often the most effective cure for overfitting. See **Section 2.2**.

---

## 3. Quick Diagnosis Guide

| Symptom | Likely Problem | First Fix to Try |
|---|---|---|
| Loss is `NaN` | Exploding gradients | Gradient clipping, lower $\alpha$ |
| Loss barely moves | Vanishing gradients / low $\alpha$ | Use ReLU, raise $\alpha$, check init |
| Train acc high, val acc low | Overfitting | Dropout, regularization, more data |
| Train acc low, val acc low | Underfitting | More layers, more epochs, higher $\alpha$ |
| Training is very slow | Bottleneck in data/compute | Mixed precision, more workers, GPU |
| Model doesn't generalize | Not enough data | Augmentation, transfer learning |

---

## Key Takeaway

> There is no single fix for a poorly performing neural network. **Diagnose first** — identify whether you're dealing with a gradient problem, a data problem, a capacity problem, or a regularization problem — then apply the right tool. Start simple and add complexity only when needed.